# File search over `docs/` — upload and query

Puts the repo's `docs/` folder into an OpenAI **vector store** and asks questions against
it with the hosted [file search
tool](https://developers.openai.com/api/docs/guides/tools-file-search). OpenAI does the
chunking, embedding and retrieval; nothing lands in our pgvector database.

`token_budget.ipynb` next door prices the do-it-yourself version — this is the managed one.

**This one spends money and creates state on the OpenAI account**, unlike the sizing
notebook. It is small (`docs/` is ~21k tokens, well under a cent to embed) and storage
is free under 1 GB, but the store persists until you delete it — the last cell does that.
Re-running is safe: the store is looked up by name and reused rather than duplicated.

In [ ]:
import sys
from pathlib import Path

from openai import OpenAI

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "backend"))
from config import settings  # noqa: E402

client = OpenAI(api_key=settings.openai.API_KEY)
MODEL = settings.openai.BASE_MODEL
STORE_NAME = "book-recommender-docs"

doc_paths = sorted((REPO_ROOT / "docs").rglob("*.md"))
print(f"{len(doc_paths)} files, {sum(p.stat().st_size for p in doc_paths) / 1024:.0f} KB")
for p in doc_paths:
    print(" ", p.relative_to(REPO_ROOT))

8 files, 83 KB
  docs/README.md
  docs/backlog.md
  docs/design/execution-pipeline-v1.md
  docs/design/human-in-the-loop.md
  docs/design/node-taxonomy-v1.md
  docs/design/planner-shape.md
  docs/eval-strategy.md
  docs/roadmap.md


## Upload

`upload_and_poll` uploads every file and blocks until each one is chunked and embedded, so
one call covers `files.create` + `vector_stores.files.create` + polling.

Files are sent as `(name, bytes)` pairs with the path flattened into the name
(`docs__design__node-taxonomy-v1.md`). A bare handle would upload as its basename, and the
moment this grows past `docs/` that collides — this repo has a `README.md` in nearly every
folder, and the citations below are only useful if they name one file.

In [4]:
existing = next((v for v in client.vector_stores.list() if v.name == STORE_NAME), None)
store = existing or client.vector_stores.create(name=STORE_NAME)
print(("reusing" if existing else "created"), store.id)

if not existing:
    uploads = [(str(p.relative_to(REPO_ROOT)).replace("/", "__"), p.read_bytes())
               for p in doc_paths]
    batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=store.id, files=uploads
    )
    print(batch.status, batch.file_counts)

created vs_6a6e220b698481919d78ddc6814dee57
completed FileCounts(cancelled=0, completed=8, failed=0, in_progress=0, total=8)


In [5]:
# What actually made it in. `filename` needs a files.retrieve — the vector store
# row only carries the file id.
store = client.vector_stores.retrieve(store.id)
print(f"{store.name}: {store.file_counts.completed} files, "
      f"{store.usage_bytes / 1024:.0f} KB indexed")

for f in client.vector_stores.files.list(vector_store_id=store.id):
    print(f"  {f.status:10} {client.files.retrieve(f.id).filename}")

book-recommender-docs: 8 files, 188 KB indexed
  completed  docs__backlog.md
  completed  docs__eval-strategy.md
  completed  docs__roadmap.md
  completed  docs__design__planner-shape.md
  completed  docs__design__execution-pipeline-v1.md
  completed  docs__design__human-in-the-loop.md
  completed  docs__README.md
  completed  docs__design__node-taxonomy-v1.md


## Ask it something

Two ways in. `vector_stores.search` is raw retrieval — chunks and scores, no model, which
is what you want when checking whether retrieval itself is any good. `responses.create`
with the `file_search` tool is the full loop: the model runs the search and answers, and
`include=["file_search_call.results"]` shows what it retrieved to get there.

In [9]:
QUESTION = "Why is Retrieve_Random the only node that carries a full BooksFilter?"

hits = client.vector_stores.search(vector_store_id=store.id, query=QUESTION,
                                   max_num_results=3)
for hit in hits.data:
    text = " ".join(c.text for c in hit.content)
    print(f"[{hit.score:.3f}] {hit.filename}\n  {text}...\n")

[0.869] docs__design__node-taxonomy-v1.md
  A title paired with an author is now expressed the way every other two-dimension request
is — by composition: `Retrieve_by_Title` + `Retrieve_by_Author` + `Combine_Intersect`.
This finishes what the 2026-07-21 author split started: every retrieval node is now
single-dimension *and* single-valued, with no exceptions, so the combine tier is the only
place a plan says "and".

What this buys, beyond consistency: authorship verification becomes real. "Did Jane
Austen write Dune?" used to be a title lookup that could only ever return Dune — the
author hint had no way to contradict it. Intersected against Austen's bibliography, the
empty result *is* the "no", the same way `FindByCoAuthorsOutput`'s empty `books` answers
"did they ever write together?".

What it costs: three nodes where one used to do, and the author leg fetches a whole
bibliography to keep one book. For a common query shape ("find X by Y") that is a real
latency and token increase, a

In [7]:
response = client.responses.create(
    model=MODEL,
    input=QUESTION,
    tools=[{"type": "file_search", "vector_store_ids": [store.id],
            "max_num_results": 5}],
    include=["file_search_call.results"],
)

print(response.output_text)

print("\n--- cited ---")
for item in response.output:
    for block in getattr(item, "content", None) or []:
        for ann in getattr(block, "annotations", None) or []:
            print(" ", getattr(ann, "filename", ann.type))

# Retrieved chunks are billed as ordinary input tokens on top of the question.
print(f"\ntokens: {response.usage.input_tokens} in / "
      f"{response.usage.output_tokens} out")

The reason why `Retrieve_Random` is the only node that carries a full `BooksFilter` is because of its unique role and design in the retrieval system:

- All other retrieval nodes are strictly single-dimension and single-valued in their filters. For example, nodes like `Retrieve_by_Title`, `Retrieve_by_ISBN13`, `Retrieve_by_Author`, and `Retrieve_by_Genre` target exactly one dimension with a simple field (e.g., title string, author string, genre string). They do not carry a multi-field filter object like `BooksFilter`.

- `Retrieve_Random` is different because it does not query by a specific dimension. Instead, it performs an arbitrary random pick of a book. Since it has no single dimension to constrain the pick, it carries an optional `BooksFilter` to specify any filtering criteria as a multi-field object, effectively allowing "search within bounds" before picking randomly.

- This design choice reflects the semantics that a pure random retrieval needs a way to limit its domain by vari

## Clean up

Deleting the store does not delete the uploaded files — those stay on the account under
`purpose="assistants"` until removed separately. Both lines below do the full teardown.

In [8]:
# file_ids = [f.id for f in client.vector_stores.files.list(vector_store_id=store.id)]
# client.vector_stores.delete(store.id)
# for fid in file_ids:
#     client.files.delete(fid)